In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import PromptTemplate
import transformers
import torch
from langchain_core.output_parsers import StrOutputParser
from huggingface_hub import login
from langchain_huggingface.llms import HuggingFacePipeline
from datasets import load_dataset
from langchain_core.output_parsers import BaseOutputParser

import csv
import pickle

query_list = []
with open("dev-data-fi/miracl-v1.0-fi_topics_topics.miracl-v1.0-fi-dev.tsv") as fd:
    rd = csv.reader(fd, delimiter="\t", quotechar='"')
    for row in rd:
        query_list.append(row)

with open ('retrieve_fi', 'rb') as fp:
    itemlist = pickle.load(fp)

In [ ]:
local_llm = 'llama3'
prompt = PromptTemplate(
    template="""<|begin_of_text|><|start_header_id|>system<|end_header_id|> You are an assistant for question-answering tasks. 
    Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise and write the answer in Finnish<|eot_id|><|start_header_id|>user<|end_header_id|>
    Question: {question} 
    Context: {context} 
    Answer: <|eot_id|><|start_header_id|>assistant<|end_header_id|>""",
    input_variables=["question", "document"],
)

llm = ChatOllama(model=local_llm, temperature=0.3)

answer_list = []
rag_chain = prompt | llm | StrOutputParser()
for i in tqdm(range(len(query_list))):
    docs = '\n\n'.join([doc for doc in itemlist[i]])
    generation = rag_chain.invoke({"context": docs, "question": query_list[i]})
    answer_list.append(generation)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
local_llm = 'llama3'

llm = ChatOllama(model=local_llm, format="json", temperature=0)

prompt = PromptTemplate(
    template=""" <|begin_of_text|><|start_header_id|>system<|end_header_id|> You are a grader assessing whether 
    an answer contains all information of a set of facts. Give a score from 1 to 10 to  to indicate 
    whether the answer covers all information from a set of facts. Provide the score as a JSON with a 
    single key 'score' and no preamble or explanation. <|eot_id|><|start_header_id|>user<|end_header_id|>
    Here are the facts:
    \n ------- \n
    {documents} 
    \n ------- \n
    Here is the answer: {generation}  <|eot_id|><|start_header_id|>assistant<|end_header_id|>""",
    input_variables=["generation", "documents"],
)
evaluator = prompt | llm | JsonOutputParser()
llm_score = []
error_list = []

for i in tqdm(range(len(answer_list))):
    docs = '\n\n'.join([doc for doc in itemlist[i]])
    try:
        out = evaluator.invoke({"documents": docs, "generation": answer_list[i]})
        llm_score.append(out['score'])
    except:
        error_list.append(i)

In [ ]:
with open('answer_fi', 'wb') as fp:
    pickle.dump(answer_list, fp)
with open('error_fi', 'wb') as fp:
    pickle.dump(error_list, fp)
with open('llm_score_fi', 'wb') as fp:
    pickle.dump(llm_score, fp)
with open('retrieve_fi', 'wb') as fp:
    pickle.dump(itemlist, fp)

In [ ]:
import pickle
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from numpy.linalg import norm
import bert_score

with open ('reevaluate/answer_fi', 'rb') as fp:
    answer_list = pickle.load(fp)
with open ('reevaluate/error_fi', 'rb') as fp:
    error_list = pickle.load(fp)
with open ('reevaluate/llm_score_fi', 'rb') as fp:
    llm_score = pickle.load(fp)
with open ('reevaluate/retrieve_fi', 'rb') as fp:
    itemlist = pickle.load(fp)

vectorizer = TfidfVectorizer()
all_texts = []

for i in range(len(answer_list)):
    all_texts.extend([(item) for item in itemlist[i]])
    all_texts.append((answer_list[i]))
tfidf_matrix = vectorizer.fit_transform(all_texts)


model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

index_score_list = []
cosine_score_list = []

def quality_index(embed, answer):
    q = []
    lambdas = []

    for e in embed:
        temp_val = np.abs(e - answer)
        q.append(temp_val)
    q = np.array(q)

    with np.errstate(divide='ignore', invalid='ignore'):
        def safe_divide(a, b, num_info):
            result = np.divide(a, b, out=np.zeros_like(a, dtype=float), where=(b!=0))
            result[np.isnan(result)] = 0
            return result

        for e in embed:
            temp_val = safe_divide(e.tolist(), sum(embed).tolist(), len(embed))
            lambdas.append(temp_val)
        lambdas = np.array(lambdas)

    c = np.maximum.reduce(embed)
    C = c / np.sum(c)
    output = np.dot(C.T, np.sum(lambdas * q, axis =0))
    return output

def cosine_sim(x, y):
    cosine = np.dot(x,y)/(norm(x)*norm(y))
    return cosine

temp = 0
for i in tqdm(range(len(answer_list))):
    if i in error_list: continue
    docs = itemlist[i]

    embed = [model.encode(doc) for doc in docs]
    answer = model.encode(answer_list[i])

    c = [cosine_sim(e, answer) for e in embed]
    try:
      cosine = sum(c) / len(c)
    except:
      print(c)
      print(itemlist[i])
      exit()
    cosine_score_list.append(cosine)

    embed = []
    for j in range(len(docs)):
        embed.append(tfidf_matrix[temp + j].toarray().flatten())
    answer = tfidf_matrix[temp + len(docs)].toarray().flatten()
    temp = temp + len(docs) + 1
    q = quality_index(embed, answer)
    index_score_list.append(q)


ref = []
for i in range(len(answer_list)):
    if i in error_list: continue
    docs = ' '.join([doc for doc in itemlist[i]])
    ref.append(docs)

cand = []
for i in range (len(answer_list)):
    if i in error_list: continue
    cand.append(answer_list[i])
P, R, F1 = bert_score.score(cand, ref, lang="fi", verbose=True)
bert_score_list = F1.tolist()
data = pd.DataFrame({
    'LLM': llm_score,
    'Bert score': bert_score_list,
    'Cosine similarity': cosine_score_list,
    'Quality index': index_score_list
})

correlation_matrix = data.corr()


print(correlation_matrix)

print(data.mean())

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/32 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/19 [00:00<?, ?it/s]

done in 17.81 seconds, 65.42 sentences/sec
                        LLM  Bert score  Cosine similarity  Quality index
LLM                1.000000    0.561068           0.508556      -0.008180
Bert score         0.561068    1.000000           0.713409       0.003241
Cosine similarity  0.508556    0.713409           1.000000       0.019334
Quality index     -0.008180    0.003241           0.019334       1.000000
LLM                  2.912017
Bert score           0.582357
Cosine similarity    0.371393
Quality index        0.273696
dtype: float64
